# OpenSetDGA – Multi-seed training (seeds 99 & 314)
Runs `run_multi_seed.py --seeds 99 314 --device cuda` and zips results for download.

**Steps:**
1. Install dependencies
2. Clone source code from GitHub
3. Download processed dataset from HuggingFace
4. Train all models for seeds 99 and 314
5. Zip `baseline_out/` for download

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightgbm', 'huggingface_hub', 'scikit-learn', 'tldextract'], check=True)
print('Packages ready.')

In [ ]:
# ── 2. Clone source code ──────────────────────────────────────────────────────
import os
from pathlib import Path

WORKDIR = Path('/kaggle/working/OpenSetDGA-Detection')

if not WORKDIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/quanturong/OpenSetDGA-Detection.git',
         str(WORKDIR)],
        check=True
    )
    print('Repo cloned.')
else:
    print('Repo already exists, skipping clone.')

os.chdir(WORKDIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── 3. Download processed dataset from HuggingFace ───────────────────────────
from huggingface_hub import snapshot_download

DATA_DIR = WORKDIR / 'data' / 'processed'

if not (DATA_DIR / 'known' / 'train.csv').exists():
    print('Downloading dataset from HuggingFace...')
    snapshot_download(
        repo_id='ThanhPhuongtphz/OpenSetDGA-Detection',
        repo_type='dataset',
        local_dir=str(DATA_DIR),
        ignore_patterns=['*.git*', '*.md', '*.txt'],
    )
    print('Download complete.')
else:
    print('Data already present, skipping download.')

# Verify expected files exist
expected = [
    DATA_DIR / 'known' / 'train.csv',
    DATA_DIR / 'known' / 'val.csv',
    DATA_DIR / 'known' / 'test_known.csv',
    DATA_DIR / 'unknown_family' / 'test_unknown_family.csv',
    DATA_DIR / 'unknown_ood' / 'test_unknown_ood.csv',
]
all_ok = True
for p in expected:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  {status}  {p.relative_to(WORKDIR)}')
    if not p.exists():
        all_ok = False

assert all_ok, 'Some data files missing — check HuggingFace dataset structure above.'

In [ ]:
# ── 4. Run seeds 99 and 314 ───────────────────────────────────────────────────
import os
os.environ['PYTHONIOENCODING'] = 'utf-8'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

result = subprocess.run(
    [sys.executable, 'src/run_multi_seed.py',
     '--seeds', '99', '314',
     '--device', device,
     '--run_dir', 'data/processed'],
    cwd=str(WORKDIR)
)
print(f'\nExit code: {result.returncode}')

In [ ]:
# ── 5. Verify results ─────────────────────────────────────────────────────────
import json

for seed in [99, 314]:
    print(f'\n--- Seed {seed} ---')
    for model in ['lgbm_binary', 'lgbm_multi', 'cnn', 'bilstm', 'bilstm_oe', 'extra_ood']:
        p = WORKDIR / 'baseline_out' / f'{model}_s{seed}' / 'results.json'
        if p.exists():
            print(f'  OK  {model}_s{seed}')
        else:
            print(f'  MISSING  {model}_s{seed}')

In [ ]:
# ── 6. Zip results for download ───────────────────────────────────────────────
import shutil

dirs_to_zip = []
for seed in [99, 314]:
    for model in ['lgbm_binary', 'lgbm_multi', 'cnn', 'bilstm', 'bilstm_oe', 'extra_ood']:
        d = WORKDIR / 'baseline_out' / f'{model}_s{seed}'
        if d.exists():
            dirs_to_zip.append(str(d))

zip_path = '/kaggle/working/results_s99_s314'
shutil.make_archive(zip_path, 'zip', WORKDIR / 'baseline_out')
print(f'Saved: {zip_path}.zip')
print('Download this file from the Output tab on the right.')